# scVI default - neurips_bmmc

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, TRAJ, ISOLISI, BRAS, KBET, CPCR

This notebook is intentionally unexecuted.


In [1]:
import gc
import inspect
import json
import os
import sys
from pathlib import Path

os.environ["PYTHONNOUSERSITE"] = "1"
PROJECT_ROOT = (
    "/ubc/cs/research/beaver/projects/"
    "Single-cell-integration-metrics"
)
os.environ["R_HOME"] = (
    f"{PROJECT_ROOT}/CondaEnv/scvi_py310/lib/R"
)
os.environ["R_LIBS_SITE"] = (
    f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R/library"
)
sys.path.insert(0, f"{PROJECT_ROOT}/scib")

OPTIMIZER_PATH = Path(
    PROJECT_ROOT,
    "scripts",
    "optuna_multi_metrics",
    "optuna_scvi",
    "optimize_scvi_multi_metrics.py",
)
sys.path.insert(0, str(OPTIMIZER_PATH.parent))

import anndata as ad
import numpy as np
import pandas as pd
import torch
import optimize_scvi_multi_metrics as scvi_workflow

assert Path(scvi_workflow.__file__).resolve() == OPTIMIZER_PATH


In [2]:
DATASET_NAME = 'neurips_bmmc'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEYS = ('batch',)
BATCH_KEY = BATCH_KEYS[0]
LABEL_KEY = 'cell_type'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'TRAJ', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_NEIGHBORS = 15
USE_HVG = True
N_TOP_HVG = 2000
MAX_EPOCHS = 400

EMBED_KEY = "X_scvi"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "scvi_metric_batch"
METRIC_LABEL_KEY = "scvi_metric_label"

assert DATASET_NAME == 'neurips_bmmc'
assert BATCH_KEYS == ('batch',)
assert LABEL_KEY == 'cell_type'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'TRAJ', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_keys": list(BATCH_KEYS),
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
        "max_epochs": MAX_EPOCHS,
    },
    name="configuration",
)


dataset                                            neurips_bmmc
data_path     /ubc/cs/research/beaver/projects/Single-cell-i...
batch_keys                                              [batch]
label_key                                             cell_type
metrics       [BAMI, CASW, SCGRAPHM, TRAJ, ISOLISI, BRAS, KB...
seeds                            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
use_hvg                                                    True
n_top_hvg                                                  2000
max_epochs                                                  400
Name: configuration, dtype: object

In [3]:
scvi_workflow.DATA_PATH = DATA_PATH
scvi_workflow.DATASET_NAME = DATASET_NAME
scvi_workflow.BATCH_KEYS = list(BATCH_KEYS)
scvi_workflow.BATCH_KEY = BATCH_KEY
scvi_workflow.LABEL_KEY = LABEL_KEY
scvi_workflow.USE_HVG = USE_HVG
scvi_workflow.N_TOP_HVG = N_TOP_HVG
scvi_workflow.MAX_EPOCHS = MAX_EPOCHS
scvi_workflow.N_NEIGHBORS = N_NEIGHBORS
scvi_workflow.EMBED_KEY = EMBED_KEY
scvi_workflow.CLUSTER_KEY = CLUSTER_KEY
scvi_workflow.METRIC_BATCH_KEY = METRIC_BATCH_KEY
scvi_workflow.METRIC_LABEL_KEY = METRIC_LABEL_KEY

scvi_workflow.USE_BAMI = "BAMI" in SELECTED_METRICS
scvi_workflow.USE_CASW = "CASW" in SELECTED_METRICS
scvi_workflow.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
scvi_workflow.USE_TRAJ = "TRAJ" in SELECTED_METRICS
scvi_workflow.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
scvi_workflow.USE_BRAS = "BRAS" in SELECTED_METRICS
scvi_workflow.USE_KBET = "KBET" in SELECTED_METRICS
scvi_workflow.USE_CPCR = "CPCR" in SELECTED_METRICS

scvi_workflow.validate_configuration(SELECTED_METRICS)
scvi_workflow.validate_dataset_contents()

pd.Series(
    {
        "optimizer": str(OPTIMIZER_PATH),
        "scvi_version": scvi_workflow.scvi.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    name="preflight",
)


Dataset preflight: n_obs=90261 n_batches=12 n_labels=45 counts_layer=True
TRAJ preflight: usable_pseudotime=51407 unique_values=51251
ISOLISI preflight: min_batch_coverage=1 isolated_labels=['CD8+ T naive CD127+ CD26- CD101-', 'T prog cycling', 'cDC1']
CPCR preflight: eligible_labels=40


optimizer         /ubc/cs/research/beaver/projects/Single-cell-i...
scvi_version                                                  1.3.3
cuda_available                                                 True
Name: preflight, dtype: object

In [4]:
constructor_signature = inspect.signature(
    scvi_workflow.scvi.model.SCVI.__init__
)
training_signature = inspect.signature(
    scvi_workflow.scvi.train.TrainingPlan.__init__
)
assert scvi_workflow.scvi.__version__ == "1.3.3"
assert constructor_signature.parameters["n_latent"].default == 10
assert constructor_signature.parameters["n_hidden"].default == 128
assert constructor_signature.parameters["n_layers"].default == 1
assert constructor_signature.parameters["dropout_rate"].default == 0.1
assert training_signature.parameters["lr"].default == 0.001

SCVI_PARAMS = {'n_latent': 10, 'n_hidden': 128, 'n_layers': 1, 'dropout_rate': 0.1, 'lr': 0.001}
assert tuple(SCVI_PARAMS) == ('n_latent', 'n_hidden', 'n_layers', 'dropout_rate', 'lr')

pd.Series(
    {
        **SCVI_PARAMS,
        "gene_likelihood": "zinb",
        "dispersion": "gene",
        "latent_distribution": "normal",
        "early_stopping": True,
        "max_epochs": MAX_EPOCHS,
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
    },
    name="default_setting",
)

def run_scvi_setting(adata, seed):
    return scvi_workflow.run_scvi(
        adata,
        n_latent=SCVI_PARAMS["n_latent"],
        n_hidden=SCVI_PARAMS["n_hidden"],
        n_layers=SCVI_PARAMS["n_layers"],
        dropout_rate=SCVI_PARAMS["dropout_rate"],
        lr=SCVI_PARAMS["lr"],
        seed=int(seed),
    )


In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    result = scvi_workflow.compute_joint_metrics(adata_raw, adata_int)
    if tuple(result) != (
        *SELECTED_METRICS,
        "bio_score",
        "batch_score",
        "joint_score",
    ):
        raise ValueError(f"Unexpected metric result keys: {tuple(result)}")
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_scvi_setting(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


[rank: 0] Seed set to 0


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.7000720955510094


optimised clustering against scvi_metric_label
optimal cluster resolution: 1.4
optimal score: 0.7226595673947611
BRAS...


seed           9.000000
BAMI           0.722660
CASW           0.557909
SCGRAPHM       0.746690
TRAJ           0.619914
ISOLISI        0.956847
BRAS           0.772117
KBET           0.326854
CPCR           0.456533
bio_score      0.720804
batch_score    0.518502
joint_score    0.639883
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,TRAJ,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.721482,0.558109,0.735807,0.624181,0.961078,0.768724,0.327382,0.474174,0.720131,0.523427,0.641449
1,1,0.717195,0.553607,0.762788,0.652484,0.958343,0.766303,0.326651,0.470383,0.728883,0.521112,0.645775
2,2,0.711038,0.558200,0.747529,0.453243,0.960648,0.770693,0.324683,0.469933,0.686132,0.521769,0.620387
3,3,0.726424,0.559259,0.755424,0.657536,0.956065,0.765232,0.316450,0.447645,0.730942,0.509775,0.642475
4,4,0.723203,0.560082,0.728157,0.596588,0.956978,0.768670,0.324690,0.466486,0.713001,0.519949,0.635780
5,5,0.725540,0.559113,0.733364,0.683735,0.956367,0.770086,0.318591,0.478967,0.731624,0.522548,0.647993
6,6,0.722319,0.557497,0.742714,0.549981,0.959354,0.763297,0.316406,0.463818,0.706373,0.514507,0.629627
7,7,0.720211,0.559504,0.746459,0.618286,0.958826,0.765245,0.324331,0.464535,0.720657,0.518037,0.639609
8,8,0.724724,0.557670,0.746006,0.606570,0.952797,0.764156,0.328851,0.459609,0.717553,0.517539,0.637547
9,9,0.722660,0.557909,0.746690,0.619914,0.956847,0.772117,0.326854,0.456533,0.720804,0.518502,0.639883


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.721479,0.004307,0.711038,0.726424
CASW,0.558095,0.001704,0.553607,0.560082
SCGRAPHM,0.744494,0.009715,0.728157,0.762788
TRAJ,0.606252,0.061722,0.453243,0.683735
ISOLISI,0.957730,0.002329,0.952797,0.961078
BRAS,0.767452,0.002856,0.763297,0.772117
KBET,0.323489,0.004388,0.316406,0.328851
CPCR,0.465208,0.008590,0.447645,0.478967
bio_score,0.717610,0.012905,0.686132,0.731624
batch_score,0.518717,0.003912,0.509775,0.523427
